**Importing Libraries**

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

**Define Paths**

In [ ]:
DATA_DIR = Path("https://github.com/shubhankarrana-pixel/V02Max-LightBGM/tree/main/Data/Rawfiles")

CVX_PATH = DATA_DIR / "CVX.xpt"
DEMO_PATH = DATA_DIR / "DEMO.xpt"
BMX_PATH = DATA_DIR / "BMX.xpt"

OUTPUT_DIR = Path("https://github.com/shubhankarrana-pixel/V02Max-LightBGM/tree/main/Data/Processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

**Load Data**

In [ ]:
cvx = pd.read_sas(CVX_PATH)

demo = pd.read_sas(DEMO_PATH)

bmx = pd.read_sas(BMX_PATH)

**Dataset Inspection**

In [ ]:
datasets = {
    "CVX": cvx,
    "DEMO": demo,
    "BMX": bmx
}

for name, df in datasets.items():

    print("="*70)

    print(name)

    print("="*70)

    print("Shape:", df.shape)

    print()

    print(df.info())

    print()

    print(df.head())

**Variable Selection**

In [ ]:
cvx_cols = [
    "SEQN",

    "CVDVOMAX",

    "CVDFITLV",

    "CVDEXSTS",

    "CVDPMHR",

    "CVDWTIM",

    "CVDS1TIM",

    "CVDS2TIM"
]

cvx = cvx[cvx_cols]

In [ ]:
demo_cols = [
    "SEQN",

    "RIDAGEYR",

    "RIAGENDR"
]

demo = demo[demo_cols]

In [ ]:
bmx_cols = [

    "SEQN",

    "BMXHT",

    "BMXWT",

    "BMXBMI"
]

bmx = bmx[bmx_cols]

**Merge**

In [ ]:
df = cvx.merge(
    demo,
    on="SEQN",
    how="left"
)

df = df.merge(
    bmx,
    on="SEQN",
    how="left"
)

print(df.shape)

**Rename Variables**

In [ ]:
rename_dict = {

    "RIDAGEYR":"Age",

    "RIAGENDR":"Sex",

    "BMXHT":"Height_cm",

    "BMXWT":"Weight_kg",

    "BMXBMI":"BMI",

    "CVDVOMAX":"VO2max",

    "CVDFITLV":"Fitness_Level",

    "CVDEXSTS":"Exercise_Status",

    "CVDPMHR":"Max_HR",

    "CVDWTIM":"Warmup_Time",

    "CVDS1TIM":"Stage1_Time",

    "CVDS2TIM":"Stage2_Time"
}

df.rename(columns=rename_dict,
          inplace=True)

**Encode Sex**

In [ ]:
df["Sex"] = df["Sex"].map({

    1:"Male",

    2:"Female"
})

**DATA TYPES**

In [ ]:
df.dtypes
numeric_cols = [

    "Age",

    "Height_cm",

    "Weight_kg",

    "BMI",

    "VO2max",

    "Max_HR"
]

df[numeric_cols] = df[numeric_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

**Duplicate Check**

In [ ]:
duplicates = df.duplicated("SEQN").sum()

print("Duplicate Participants:", duplicates)

**Missing Value Analysis**

In [ ]:
missing = pd.DataFrame({

    "Missing Count": df.isna().sum(),

    "Missing %":

    (df.isna().mean()*100).round(2)

})

missing.sort_values(

    "Missing %",
    ascending=False
)
missing.to_csv(
    OUTPUT_DIR/"missing_report.csv"
)

**Missing Value Treatment**

In [ ]:
from sklearn.impute import SimpleImputer

median_cols = [

    "Height_cm",

    "Weight_kg",

    "BMI",

    "Max_HR"
]

median_imputer = SimpleImputer(
    strategy="median"
)

df[median_cols] = median_imputer.fit_transform(
    df[median_cols]
)
cat_cols = [

    "Sex",

    "Fitness_Level",

    "Exercise_Status"
]

cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

df[cat_cols] = cat_imputer.fit_transform(
    df[cat_cols]
)

**Physiological Range Check**

In [ ]:
df["Flag_Invalid_VO2"] = (
    (df["VO2max"] < 10) |
    (df["VO2max"] > 90)
)

df["Flag_Invalid_BMI"] = (
    (df["BMI"] < 10) |
    (df["BMI"] > 60)
)

df["Flag_Invalid_Height"] = (
    (df["Height_cm"] < 120) |
    (df["Height_cm"] > 230)
)

df["Flag_Invalid_Weight"] = (
    (df["Weight_kg"] < 25) |
    (df["Weight_kg"] > 250)
)

**BMI Category**

In [ ]:
df["BMI_Category"] = pd.cut(
    df["BMI"],
    bins=[0,18.5,25,30,100],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

**Cardiovascular Features**

In [ ]:
df["Absolute_VO2"] = (
    df["VO2max"] *
    df["Weight_kg"]
)/1000
df["Height_Weight_Ratio"] = (
    df["Height_cm"] /
    df["Weight_kg"]
)
df["Predicted_HRmax"] = (
    220 -
    df["Age"]
)
df["Relative_HR_Response"] = (
    df["Max_HR"] /
    df["Predicted_HRmax"]
)

**Saving the Processed Dataset**

In [ ]:
df.to_csv(
    OUTPUT_DIR / "cleaned_dataset.csv",
    index=False
)

print("Dataset saved successfully.")